In [11]:
import cv2
import numpy as np
import glob

# Camera Calibration

## Finding Corners

In [5]:
chessBoardSize = (24,17)
frameSize = (1024,768)

In [9]:
criteria = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, .001)

In [15]:
objp = np.zeros((chessBoardSize[0]*chessBoardSize[1], 3), np.float32)
objp[:,:2] = np.mgrid[0:chessBoardSize[0], 0:chessBoardSize[1]].T.reshape(-1,2)

In [17]:
objPoints = []
imgPoints = []

In [23]:
images = glob.glob('cameraCalibration*.png')

for image in images:
    print(image)
    img = cv2.imread(image)
    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    ret, corners = cv2.findChessboardCorners(grey, chessBoardSize, None)

    if ret == True:
        objPoints.append(objp)
        corner2 = cv2.cornerSubPix(grey, corners, (9,9), (-1,-1), criteria)
        imgPoints.append(corners)
        print(len(corner2))
        cv2.drawChessboardCorners(img, chessBoardSize, corner2, ret)
        cv2.imshow('corner found', img)
        cv2.waitKey(2000)

cv2.destroyAllWindows()

cameraCalibration1.png
408
cameraCalibration2.png
408
cameraCalibration3.png
408
cameraCalibration4.png
408
cameraCalibration5.png
408


## Calibrating Cameras

In [30]:
ret, cameraMaterix, dist, rvect, tvect = cv2.calibrateCamera(objPoints, imgPoints, frameSize, None, None)
print("Camera Calibration", ret)
print("Camera Matrix \n", cameraMaterix)
print("Distortion Parameter \n", dist)
print("Rotation Vector \n", rvect)
print("Translation Vector \n", tvect)

Camera Calibration 4.113234639591478
Camera Matrix 
 [[857.35296726   0.         469.77891592]
 [  0.         857.5799464  378.54273185]
 [  0.           0.           1.        ]]
Camera Distortion 
 [[-0.23071588  0.05938638  0.0145262  -0.00133216  0.11729316]]
Rotation Vector 
 (array([[ 0.00407198],
       [ 0.09186078],
       [-0.01262096]]), array([[-0.04661902],
       [ 0.45036067],
       [-0.06143955]]), array([[-0.70607686],
       [ 0.04299497],
       [ 0.05266104]]), array([[-0.02734145],
       [ 0.05652998],
       [-1.59994223]]), array([[ 0.00407198],
       [ 0.09186078],
       [-0.01262096]]), array([[ 0.00407198],
       [ 0.09186078],
       [-0.01262096]]), array([[-0.04661902],
       [ 0.45036067],
       [-0.06143955]]), array([[-0.70607686],
       [ 0.04299497],
       [ 0.05266104]]), array([[-0.02734145],
       [ 0.05652998],
       [-1.59994223]]), array([[ 0.00407198],
       [ 0.09186078],
       [-0.01262096]]), array([[ 0.00407198],
       [ 0.0918

# Undistortion

In [33]:
img = cv2.imread('cameraCalibration3.png')
h,w = img.shape[:2]
newCameraMatrix, roi = cv2.getOptimalNewCameraMatrix(cameraMaterix, dist, (w,h), 1, (w,h))

### Undistort

In [38]:
dst = cv2.undistort(img, cameraMaterix, dist, None, newCameraMatrix)
# cropping Image
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv2.imwrite('calibResult_on_cameraCalibration3.png', dst)

True

### Undistort using ReMapping

In [45]:
mapx, mapy = cv2.initUndistortRectifyMap(cameraMaterix, dist, None, newCameraMatrix, (w,h), 5)
dst = cv2.remap(img, mapx, mapy, cv2.INTER_LANCZOS4)
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv2.imwrite('calibResult_on_cameraCalibration3_using_remap.png', dst)

True